In [8]:
%pip install -qU \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-chroma \
    sentence-transformers \
    transformers \
    accelerate \
    pypdf \
    chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.

In [9]:
import os
import torch

from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM

/tmp/ipykernel_1381/966995845.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**1. Upload PDF files**

In [16]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Specify exact file path(s)
pdf_paths = ['/content/drive/MyDrive/Episode 04 - Protein.pdf', '/content/drive/MyDrive/Episode 03 - RNA Structure.pdf',
            '/content/drive/MyDrive/Episode 02 - DNA Structure and Replication.pdf', '/content/drive/MyDrive/Episode 00 - Course Info.pdf',
            '/content/drive/MyDrive/Episode 01 - Structure of Cell.pdf'

]   # ← Change this
pdf_files = []
for pdf_path in pdf_paths:
    if os.path.exists(pdf_path):
        pdf_files.append(os.path.basename(pdf_path))
    else:
        print(f"⚠️ File not found: {pdf_path}")

print("✅ Found PDFs:", pdf_files)
print(f"Total PDFs ready: {len(pdf_files)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Found PDFs: ['Episode 04 - Protein.pdf', 'Episode 03 - RNA Structure.pdf', 'Episode 02 - DNA Structure and Replication.pdf', 'Episode 00 - Course Info.pdf', 'Episode 01 - Structure of Cell.pdf']
Total PDFs ready: 5


**2. Load PDFs**

In [18]:
documents = []

for pdf_path in pdf_paths:
    if not os.path.exists(pdf_path):
        continue

    loader = PyPDFLoader(pdf_path)
    loaded_pages = loader.load()

    for doc in loaded_pages:
        doc.metadata['file_name'] = os.path.basename(pdf_path)

    documents.extend(loaded_pages)

documents = [doc for doc in documents if len(doc.page_content.strip()) >= 20]

print(f"Successfully loaded {len(documents)} pages from {len(pdf_files)} documents")

Successfully loaded 215 pages from 5 documents


In [19]:
documents

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with Beamer class', 'creationdate': '2025-11-11T00:43:18+00:00', 'author': 'Ali Sharifi-Zarchi, Somayyeh Kouhi', 'title': 'Protein', 'subject': '', 'keywords': '', 'moddate': '2025-11-11T00:43:18+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'source': '/content/drive/MyDrive/Episode 04 - Protein.pdf', 'total_pages': 54, 'page': 0, 'page_label': '1', 'file_name': 'Episode 04 - Protein.pdf'}, page_content='Protein\nAli Shariﬁ-Zarchi Somayyeh Kouhi\nSharif University of Technology\nNovember 11, 2025\nAli Shariﬁ-Zarchi, Somayyeh Kouhi (SUT) November 11, 2025 1 / 54'),
 Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with Beamer class', 'creationdate': '2025-11-11T00:43:18+00:00', 'author': 'Ali Sharifi-Zarchi, Somayyeh Kouhi', 'title': 'Protein', 'subject': '', 'keywords': '', 'moddate': '2025-11-11T00:43:18+00:00

**3. Split documents**

In [20]:
text_splitters = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""],
)

splits = text_splitters.split_documents(documents)
print(f"Successfully split into {len(splits)} chunks")

Successfully split into 219 chunks


**4. Local multilingual embeddings**

In [26]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

embeddings = HuggingFaceEmbeddings(
    model_name='intfloat/multilingual-e5-large',
    model_kwargs={"device": device},
    encode_kwargs={"batch_size": 32}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

**5. Chroma vector database**

In [27]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory='/content/chroma_db',
    collection_name='bio_slides'
)

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

print('vector database created')

vector database created


**6. Load local LLM**

In [29]:
model_name = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto'
  )

print('llm load')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

llm load


**7. RAG question-answering function**

In [31]:
def ask_slides(question, k=5, max_new_tokens=400):
  docs = vectorstore.similarity_search(question, k=k)
  context_parts = []
  sources = []

  for i, doc in enumerate(docs):
    file_name = doc.metadata['file_name']
    page = doc.metadata['page'] + 1

    context_parts.append(f"[Source {i+1}: {file_name}, page {page}]\n"
                         f"{doc.page_content}")

    sources.append((file_name, page))

  context = '\n\n'.join(context_parts)
  messages = [
      {
          "role" : "system",
          "content" : """
            فقط براساس متن اسلایدها پاسخ بده.
            اگر پاسخ در اسلایدها وجود ندارد، بگو پاسخ در اسلایدها پیدا نشد.
            پاسخ را دقیق و آموزشی بنویس و به شماره منابع اشاره کن.
            """,
      },
      {
         "role" : "user",
          "content": f"""
          متن اسلایدها:
          {context}
          سؤال:
          {question}
          """,
      }
  ]

  prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
  inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=6000
    ).to(model.device)

  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=max_new_tokens,
          do_sample=False,
          repetition_penalty=1.05,
          pad_token_id=tokenizer.eos_token_id
      )
  new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
  answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )
  print(answer)
  print("\nSources:")
  for i, (filename, page) in enumerate(sources, start=1):
      print(f"{i}. {filename}, page {page}")

  return answer


In [32]:
ask_slides("موضوع اصلی جلسه اول چیست؟")

موضوع اصلی جلسه اول یک فصل در مورد "نقطه‌بندی دیان" است.

Sources:
1. Episode 02 - DNA Structure and Replication.pdf, page 2
2. Episode 04 - Protein.pdf, page 54
3. Episode 01 - Structure of Cell.pdf, page 54
4. Episode 00 - Course Info.pdf, page 3
5. Episode 02 - DNA Structure and Replication.pdf, page 67


'موضوع اصلی جلسه اول یک فصل در مورد "نقطه\u200cبندی دیان" است.'

In [33]:
ask_slides("who is mohsen piri?")

Mohsen Piri is the Captain of the teaching team mentioned in the provided information.

Sources:
1. Episode 00 - Course Info.pdf, page 3
2. Episode 04 - Protein.pdf, page 1
3. Episode 04 - Protein.pdf, page 42
4. Episode 04 - Protein.pdf, page 31
5. Episode 04 - Protein.pdf, page 15


'Mohsen Piri is the Captain of the teaching team mentioned in the provided information.'

In [34]:
ask_slides("what is the difrence between Eukaryotes vs Prokaryotes? explain more")

The main difference between eukaryotes and prokaryotes lies in their cellular organization and the processes they use to carry out life functions.

### Eukaryotic Cells:
- **Nucleus**: Contains genetic material in the form of chromosomes, which are organized into discrete units called chromosomes.
- **Complex Organelles**: Have specialized structures such as mitochondria, chloroplasts, and endoplasmic reticulum.
- **Internal Compartmentalization**: The cell has distinct compartments that perform specific functions, allowing for efficient regulation of various processes.
- **Transcription and Translation**: Transcription occurs in the nucleus, while translation takes place in the cytoplasm. This separation allows for greater control over gene expression.
- **Promoter Structures**: Eukaryotic promoters have more complex structures, including the TATA box, which helps in the binding of transcription factors.
- **Processing Steps**: Eukaryotic pre-mRNA undergoes additional processing steps

'The main difference between eukaryotes and prokaryotes lies in their cellular organization and the processes they use to carry out life functions.\n\n### Eukaryotic Cells:\n- **Nucleus**: Contains genetic material in the form of chromosomes, which are organized into discrete units called chromosomes.\n- **Complex Organelles**: Have specialized structures such as mitochondria, chloroplasts, and endoplasmic reticulum.\n- **Internal Compartmentalization**: The cell has distinct compartments that perform specific functions, allowing for efficient regulation of various processes.\n- **Transcription and Translation**: Transcription occurs in the nucleus, while translation takes place in the cytoplasm. This separation allows for greater control over gene expression.\n- **Promoter Structures**: Eukaryotic promoters have more complex structures, including the TATA box, which helps in the binding of transcription factors.\n- **Processing Steps**: Eukaryotic pre-mRNA undergoes additional process

In [35]:
print(device)

cpu


In [36]:
ask_slides("فرآیند ترجمه تو ساخت پروتین چه قدم هایی داره؟ قدم به قدم توضیحشون بده")

فرآیند ترجمه تو ساخت پروتین چه قدم هایی داره؟ قدم به قدم توضیحشون بده

1. قدم اول: ترکیب ترکیبی از ترکیب‌های اسید اسیدی با استفاده از ترکیب‌های اسید اسیدی مرتبط با ترکیب‌های اسید اسیدی قبلی. این ترکیب‌ها به صورت گروه‌های اسید اسیدی مرتبط با یکدیگر ترکیب می‌شوند.

2. قدم دوم: ترکیب‌های اسید اسیدی مرتبط با یکدیگر را به صورت گروه‌های اسید اسیدی مرتبط با یکدیگر ترکیب می‌کند تا ترکیب‌های اسید اسیدی مرتبط با یکدیگر را تولید کند.

3. قدم سوم: ترکیب‌های اسید اسیدی مرتبط با یکدیگر را به صورت گروه‌های اسید اسیدی مرتبط با یکدیگر ترکیب می‌کند تا ترکیب‌های اسید اسیدی مرتبط با یکدیگر را تولید کند.

4. قدم چهارم: ترکیب‌های اس

Sources:
1. Episode 04 - Protein.pdf, page 24
2. Episode 04 - Protein.pdf, page 2
3. Episode 04 - Protein.pdf, page 11
4. Episode 04 - Protein.pdf, page 30
5. Episode 02 - DNA Structure and Replication.pdf, page 63


'فرآیند ترجمه تو ساخت پروتین چه قدم هایی داره؟ قدم به قدم توضیحشون بده\n\n1. قدم اول: ترکیب ترکیبی از ترکیب\u200cهای اسید اسیدی با استفاده از ترکیب\u200cهای اسید اسیدی مرتبط با ترکیب\u200cهای اسید اسیدی قبلی. این ترکیب\u200cها به صورت گروه\u200cهای اسید اسیدی مرتبط با یکدیگر ترکیب می\u200cشوند.\n\n2. قدم دوم: ترکیب\u200cهای اسید اسیدی مرتبط با یکدیگر را به صورت گروه\u200cهای اسید اسیدی مرتبط با یکدیگر ترکیب می\u200cکند تا ترکیب\u200cهای اسید اسیدی مرتبط با یکدیگر را تولید کند.\n\n3. قدم سوم: ترکیب\u200cهای اسید اسیدی مرتبط با یکدیگر را به صورت گروه\u200cهای اسید اسیدی مرتبط با یکدیگر ترکیب می\u200cکند تا ترکیب\u200cهای اسید اسیدی مرتبط با یکدیگر را تولید کند.\n\n4. قدم چهارم: ترکیب\u200cهای اس'